# 📈 Prédiction du Prix des Actions avec LSTM

## 🎯 Objectifs d'Apprentissage

Ce challenge quotidien vous permettra de :
- Prétraiter et préparer des données de séries temporelles pour les modèles d'apprentissage automatique
- Construire et entraîner un modèle LSTM (Long Short-Term Memory) en utilisant PyTorch
- Évaluer les performances d'un modèle de régression en utilisant des métriques comme R²

## 📊 Ce que vous allez créer

- Un jeu de données prétraité pour la prédiction du prix des actions
- Un modèle LSTM entraîné pour prédire les prix futurs des actions

---

In [ ]:
# Partie 1 : Installation et Importation des Bibliothèques
# Installation des bibliothèques nécessaires
!pip install kaggle opendatasets torch scikit-learn pandas numpy matplotlib -q

# Importation des bibliothèques
import opendatasets as od
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import r2_score
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

print("✅ Toutes les bibliothèques sont installées et importées avec succès!")

# Partie 2 : Téléchargement et Chargement du Dataset
# Télécharger le dataset depuis Kaggle
dataset_url = 'https://www.kaggle.com/datasets/jacksoncrow/stock-market-dataset'
od.download(dataset_url)

# Charger les données (en utilisant un fichier d'actions spécifique, par exemple AAPL)
df = pd.read_csv('stock-market-dataset/stocks/AAPL.csv')
print(f"\\n📊 Dataset chargé avec succès! Forme: {df.shape}")
print(f"\\nPremières lignes:\\n{df.head()}")
print(f"\\nInformations sur le dataset:\\n{df.info()}")

# Partie 3 : Prétraitement des Données
# Supprimer les colonnes inutiles et créer la colonne cible
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

# Créer une colonne pour le prix de clôture du jour suivant
df['Next_Close'] = df['Close'].shift(-1)
df = df.dropna()  # Supprimer la dernière ligne qui n'a pas de valeur suivante

# Sélectionner les caractéristiques pertinentes
features = ['Open', 'High', 'Low', 'Close', 'Volume']
target = 'Next_Close'

X = df[features].values
y = df[target].values

# Normalisation des données
scaler_X = MinMaxScaler()
scaler_y = MinMaxScaler()

X_scaled = scaler_X.fit_transform(X)
y_scaled = scaler_y.fit_transform(y.reshape(-1, 1))

print(f"\\n✅ Données prétraitées! Forme X: {X_scaled.shape}, Forme y: {y_scaled.shape}")

# Partie 4 : Préparation pour l'Entraînement
# Division en ensembles d'entraînement, validation et test
train_size = int(0.7 * len(X_scaled))
val_size = int(0.15 * len(X_scaled))

X_train = X_scaled[:train_size]
y_train = y_scaled[:train_size]

X_val = X_scaled[train_size:train_size+val_size]
y_val = y_scaled[train_size:train_size+val_size]

X_test = X_scaled[train_size+val_size:]
y_test = y_scaled[train_size+val_size:]

print(f"\\n✅ Données divisées: Train={len(X_train)}, Val={len(X_val)}, Test={len(X_test)}")

# Créer une classe Dataset PyTorch personnalisée
class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

# Créer les DataLoaders
batch_size = 32

train_dataset = StockDataset(X_train, y_train)
val_dataset = StockDataset(X_val, y_val)
test_dataset = StockDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

print(f"\\n✅ DataLoaders créés avec batch_size={batch_size}")

# Partie 5 : Définition du Modèle LSTM
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super(LSTMModel, self).__init__()
        # Couche LSTM
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                           batch_first=True, dropout=dropout if num_layers > 1 else 0)

        # Dropout
        self.dropout = nn.Dropout(dropout)

        # Couches entièrement connectées
        self.fc1 = nn.Linear(hidden_size, 32)
        self.fc2 = nn.Linear(32, 1)

        # Fonction d'activation
        self.relu = nn.ReLU()

    def forward(self, x):
        # Ajouter une dimension temporelle (batch, seq_len, features)
        x = x.unsqueeze(1)

        # Passer à travers LSTM
        lstm_out, _ = self.lstm(x)

        # Prendre la dernière sortie
        lstm_out = lstm_out[:, -1, :]

        # Dropout
        x = self.dropout(lstm_out)

        # Couches entièrement connectées
        x = self.relu(self.fc1(x))
        x = self.fc2(x)

        return x

# Initialiser le modèle
input_size = len(features)
hidden_size = 64
num_layers = 2

model = LSTMModel(input_size, hidden_size, num_layers)
print(f"\\n✅ Modèle LSTM créé:")
print(model)

# Partie 6 : Entraînement et Évaluation
# Configuration de l'entraînement
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 50

# Listes pour stocker les pertes
train_losses = []
val_losses = []

print(f"\\n🚀 Début de l'entraînement pour {num_epochs} epochs...")

# Boucle d'entraînement
for epoch in range(num_epochs):
    # Mode entraînement
    model.train()
    train_loss = 0

    for X_batch, y_batch in train_loader:
        # Forward pass
        predictions = model(X_batch)
        loss = criterion(predictions, y_batch)

        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    # Mode évaluation
    model.eval()
    val_loss = 0

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            predictions = model(X_batch)
            loss = criterion(predictions, y_batch)
            val_loss += loss.item()

    # Calculer les pertes moyennes
    train_loss = train_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch+1}/{num_epochs}], Train Loss: {train_loss:.6f}, Val Loss: {val_loss:.6f}")

print("\\n✅ Entraînement terminé!")

# Évaluation sur l'ensemble de test
model.eval()
test_predictions = []
test_actuals = []

with torch.no_grad():
    for X_batch, y_batch in test_loader:
        predictions = model(X_batch)
        test_predictions.extend(predictions.numpy())
        test_actuals.extend(y_batch.numpy())

# Dénormaliser les prédictions
test_predictions = scaler_y.inverse_transform(np.array(test_predictions))
test_actuals = scaler_y.inverse_transform(np.array(test_actuals))

# Calculer le score R²
r2 = r2_score(test_actuals, test_predictions)
print(f"\\n📊 Score R² sur l'ensemble de test: {r2:.4f}")

# Visualisation des résultats
plt.figure(figsize=(15, 10))

# Graphique 1: Courbes de perte
plt.subplot(2, 1, 1)
plt.plot(train_losses, label='Perte d\\'Entraînement', color='blue')
plt.plot(val_losses, label='Perte de Validation', color='orange')
plt.title('Évolution de la Perte pendant l\\'Entraînement')
plt.xlabel('Epoch')
plt.ylabel('Perte (MSE)')
plt.legend()
plt.grid(True)

# Graphique 2: Prédictions vs Valeurs Réelles
plt.subplot(2, 1, 2)
plt.plot(test_actuals[:100], label='Valeurs Réelles', color='green', marker='o', alpha=0.7)
plt.plot(test_predictions[:100], label='Prédictions', color='red', marker='x', alpha=0.7)
plt.title(f'Prédictions LSTM vs Valeurs Réelles (R² = {r2:.4f})')
plt.xlabel('Échantillons')
plt.ylabel('Prix de Clôture ($)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

print("\\n✅ Tous les processus sont terminés avec succès!")
print(f"\\n📌 Résumé:")
print(f"   - Taille du dataset: {len(df)} échantillons")
print(f"   - Features utilisées: {features}")
print(f"   - Architecture LSTM: {num_layers} couches, {hidden_size} unités cachées")
print(f"   - Epochs d'entraînement: {num_epochs}")
print(f"   - Score R² final: {r2:.4f}")